# US 5.4 -- Human-in-the-Loop Labeling Session

After selecting target-domain samples (US 5.1--5.3), a human annotator needs to
label them.  This notebook explains how the labeling workflow is managed:

1. **Create a session folder** with the selected images copied in.
2. **Track session status** (pending, in-progress, complete).
3. **Load completed labels** back into the training pipeline.

The actual annotation happens outside the codebase (e.g., in CVAT, Label Studio,
or a custom tool).  This module manages the file-system interface between
selection and training.

## What you will learn

1. How labeling sessions are structured on disk
2. How to create a session from a selection CSV
3. How to check session status
4. How to load completed labels for training

## CLI equivalent

```bash
udm-epic5 prepare-session \
    --selection results/selection_round1.csv \
    --round 1 \
    --output sessions/round_01
```

## Prerequisites

- Python 3.12
- Install: `uv pip install -e ".[epic5]"`
- A selection CSV from US 5.3

---
## 1. Session Folder Structure

Each labeling round gets its own folder.  The structure:

```
sessions/
  round_01/
    metadata.yaml          # round number, budget, strategy, timestamp
    selection.csv          # copy of the selection CSV
    images/                # symlinks or copies of selected images
      img_0012.png
      img_0047.png
      ...
    masks/                 # empty initially; human fills these in
      img_0012.png         # binary mask (added by annotator)
      img_0047.png
      ...
    status.yaml            # tracks progress: pending -> in_progress -> complete
```

### Workflow

1. Run `prepare-session` to create the folder and copy images.
2. Annotator opens the `images/` folder in their labeling tool.
3. Annotator saves binary masks to `masks/`.
4. When done, annotator updates `status.yaml` (or the tool does it).
5. Training script reads `masks/` to get the new labels.

In [ ]:
import yaml
from pathlib import Path
from datetime import datetime

from udm_epic5.labeling.session import (
    LabelingSession,
    create_labeling_session,
    load_labeled_samples,
)

---
## 2. Creating a Labeling Session

The `create_labeling_session` function takes a selection CSV and creates the
session folder.  It copies (or symlinks) the selected images and writes
the metadata.

In [ ]:
# Create a labeling session from a selection CSV
# (In practice, this CSV comes from US 5.3)
selection_csv = "results/selection_round1.csv"
session_dir = "sessions/round_01"

session = create_labeling_session(
    selection_csv=selection_csv,
    output_dir=session_dir,
    round_number=1,
    strategy="combined",
    alpha=0.5,
)

print(f"Session created at: {session.session_dir}")
print(f"  Round:    {session.round_number}")
print(f"  Budget:   {session.budget}")
print(f"  Strategy: {session.strategy}")
print(f"  Status:   {session.status}")

In [ ]:
# Inspect the session folder
session_path = Path(session_dir)

print("Session folder contents:")
for p in sorted(session_path.rglob('*')):
    rel = p.relative_to(session_path)
    if p.is_dir():
        print(f"  {rel}/")
    else:
        print(f"  {rel}")

print()
# Read metadata
metadata_path = session_path / "metadata.yaml"
if metadata_path.exists():
    with open(metadata_path) as f:
        meta = yaml.safe_load(f)
    print("metadata.yaml:")
    for k, v in meta.items():
        print(f"  {k}: {v}")

---
## 3. Tracking Session Status

The `LabelingSession` object tracks the annotation progress by counting
how many masks exist in the `masks/` folder compared to the total budget.

In [ ]:
# Load an existing session
session = LabelingSession(session_dir)

print(f"Session status: {session.status}")
print(f"  Total images:  {session.budget}")
print(f"  Labeled so far: {session.num_labeled}")
print(f"  Remaining:     {session.num_remaining}")
print(f"  Progress:      {session.progress_pct:.1f}%")
print()

# Status transitions:
# - "pending":     session created, no masks yet
# - "in_progress": at least 1 mask exists but not all
# - "complete":    all masks present
print("Status transitions:")
print("  pending -> in_progress -> complete")

In [ ]:
# Simulate the annotator completing some labels
# (In practice, masks are created by the annotation tool)
import numpy as np
from PIL import Image

masks_dir = session_path / "masks"
images_dir = session_path / "images"

# Create fake masks for the first 5 images
image_files = sorted(images_dir.glob("*.png"))
for img_file in image_files[:5]:
    # Create a random binary mask (512x512)
    fake_mask = (np.random.rand(512, 512) > 0.8).astype(np.uint8) * 255
    mask_path = masks_dir / img_file.name
    Image.fromarray(fake_mask).save(mask_path)

# Refresh status
session.refresh()
print(f"After simulated labeling:")
print(f"  Status:   {session.status}")
print(f"  Labeled:  {session.num_labeled}")
print(f"  Progress: {session.progress_pct:.1f}%")

---
## 4. Loading Completed Labels

Once the annotator finishes (or even partially), we load the labeled
image-mask pairs back into the training pipeline.  The `load_labeled_samples`
function returns a list of `(image_path, mask_path)` tuples for all
completed annotations.

In [ ]:
# Load labeled samples from the session
labeled_samples = load_labeled_samples(session_dir)

print(f"Loaded {len(labeled_samples)} labeled samples:")
for img_path, mask_path in labeled_samples[:5]:
    print(f"  Image: {img_path}")
    print(f"  Mask:  {mask_path}")
    print()

print("These pairs are added to the training set for Active DANN (US 5.5).")

In [ ]:
# Loading labels from multiple rounds
# In a multi-round active learning loop, labels accumulate:

all_labeled = []
session_dirs = ["sessions/round_01"]  # add more rounds as they complete

for sd in session_dirs:
    if Path(sd).exists():
        samples = load_labeled_samples(sd)
        all_labeled.extend(samples)
        print(f"  {sd}: {len(samples)} samples")

print(f"\nTotal labeled target samples across all rounds: {len(all_labeled)}")
print("These are combined with the source labels for Active DANN training.")

---
## 5. CLI Usage

```bash
# Create a new labeling session
udm-epic5 prepare-session \
    --selection results/selection_round1.csv \
    --round 1 \
    --output sessions/round_01

# Check session status
udm-epic5 prepare-session --status sessions/round_01

# For round 2 (after round 1 training):
udm-epic5 prepare-session \
    --selection results/selection_round2.csv \
    --round 2 \
    --output sessions/round_02
```

### Integration with annotation tools

The session `images/` folder can be imported directly into:
- **CVAT**: create a task pointing to the images folder
- **Label Studio**: import images, export masks to the `masks/` folder
- **Custom tool**: any tool that reads PNGs and writes binary masks

---
## Summary

- Each labeling round creates a session folder with images, masks, and metadata.
- Session status is tracked automatically based on mask file count.
- `load_labeled_samples` returns image-mask pairs for training.
- Labels accumulate across rounds for the active learning loop.

**Next:** `epic5_05_active_dann.ipynb` -- train Active DANN using the
source labels plus newly acquired target labels.